In [ ]:
"""
Genetic Algorithm - CSE422 Exam-Scheduling Solution (Roulette-Wheel Selection)
==============================================================================

Schedules a set of university exams into K time slots so that the number of
CLASHES (two conflicting exams placed in the same slot) is minimised. Two exams
conflict when at least one student must sit both, so they cannot share a slot.
A schedule with 0 clashes is a perfect, conflict-free timetable.

The problem is solved with a Genetic Algorithm (GA):

    1. represent each candidate timetable as a CHROMOSOME (one slot number per exam),
    2. score each chromosome with a FITNESS function (based on the clash count),
    3. evolve a POPULATION of chromosomes using selection, crossover and mutation,
       repeating until a conflict-free timetable is found or a generation limit
       is reached.

This version uses ROULETTE-WHEEL (fitness-proportionate) selection: every
chromosome is given a slice of an imaginary wheel sized by its fitness, and a
parent is chosen by "spinning" that wheel. Fitter timetables get bigger slices
and are picked more often, but weaker ones still have a chance.

The code is written in a deliberately plain, step-by-step style (simple loops and
named intermediate variables) so each line is easy to read and to debug. Every
function and its key variables are documented with example values.
"""

import random


# ---------------------------------------------------------------------------
# 1. Input parsing
# ---------------------------------------------------------------------------

def parse_input(text):
    """
    Parse the problem input, given as a single multi-line string.

    Expected format:
        line 1        : N M        (number of exams, number of conflicting pairs)
        line 2        : K          (number of available time slots)
        next N lines  : exam_code  (one exam per line)
        next M lines  : u v        (exams u and v conflict / share students)

    Parameters
    ----------
    text : str
        The raw problem input, e.g.
        "6 8\\n3\\nCSE110\\nCSE220\\n...\\nCSE110 CSE220\\n..."

    Returns
    -------
    exams : list[str]
        Exam codes in input order; exams[i] is the exam at index i.
        Example: ['CSE110', 'CSE220', 'MAT120', 'PHY101', 'ENG101', 'ECO101']
    conflicts : list[tuple[str, str]]
        Conflicting pairs as (exam_code, exam_code).
        Example: [('CSE110', 'CSE220'), ('CSE110', 'MAT120'), ...]
    num_slots : int
        The number of available time slots, K. Example: 3
    """
    # `raw_lines` is every line of the input, still possibly with extra spaces or
    # blank lines. Example: ['', '6 8', '3', 'CSE110', ...]
    raw_lines = text.strip().splitlines()

    # `lines` keeps only the non-empty lines, each trimmed of surrounding spaces,
    # so stray blank lines in the pasted text do not break the parsing.
    # Example: ['6 8', '3', 'CSE110', ..., 'CSE110 CSE220', ...]
    lines = []
    for raw in raw_lines:
        cleaned = raw.strip()
        if cleaned != "":
            lines.append(cleaned)

    # The first line holds N and M.
    # `first_parts` example: ['6', '8']
    first_parts = lines[0].split()
    num_exams = int(first_parts[0])       # N, e.g. 6
    num_conflicts = int(first_parts[1])   # M, e.g. 8

    # The second line holds K, the number of time slots. Example: 3
    num_slots = int(lines[1])

    # The exams are the next N lines, starting at index 2.
    exam_start = 2
    exam_end = 2 + num_exams
    # `exams` example: ['CSE110', 'CSE220', 'MAT120', 'PHY101', 'ENG101', 'ECO101']
    exams = lines[exam_start:exam_end]

    # The conflicts are the next M lines, right after the exams.
    conflict_start = exam_end
    conflict_end = exam_end + num_conflicts
    # `conflict_lines` example: ['CSE110 CSE220', 'CSE110 MAT120', ...]
    conflict_lines = lines[conflict_start:conflict_end]

    # `conflicts` turns each "u v" line into a (u, v) pair of exam codes.
    conflicts = []
    for line in conflict_lines:
        parts = line.split()   # e.g. ['CSE110', 'CSE220']
        u = parts[0]           # first exam code, e.g. 'CSE110'
        v = parts[1]           # second exam code, e.g. 'CSE220'
        pair = (u, v)
        conflicts.append(pair)

    return exams, conflicts, num_slots

In [ ]:
# ---------------------------------------------------------------------------
# 2. Building the conflict edges (exam codes -> index pairs)
# ---------------------------------------------------------------------------

def build_edges(exams, conflicts):
    """
    Convert the conflicting pairs from exam codes into chromosome-index pairs.

    A chromosome stores one slot per exam *by position*, so the fitness function
    needs the conflicts expressed as indices into that chromosome rather than as
    exam codes. This translation is done once, up front.

    Parameters
    ----------
    exams : list[str]
        Exam codes in order; exams[i] is the exam at index i.
        Example: ['CSE110', 'CSE220', 'MAT120', 'PHY101', 'ENG101', 'ECO101']
    conflicts : list[tuple[str, str]]
        Conflicting pairs as exam codes.
        Example: [('CSE110', 'CSE220'), ('MAT120', 'PHY101'), ...]

    Returns
    -------
    edges : list[tuple[int, int]]
        The same conflicts as (index, index) pairs.
        Example: [(0, 1), (0, 2), (1, 2), (2, 3), (1, 3), (3, 4), (3, 5), (4, 5)]
        Here (0, 1) means exams[0] (CSE110) and exams[1] (CSE220) conflict.
    """
    # `position` maps each exam code to its index in the chromosome.
    # Example: {'CSE110': 0, 'CSE220': 1, 'MAT120': 2, 'PHY101': 3, 'ENG101': 4, 'ECO101': 5}
    position = {}
    for i in range(len(exams)):
        exam = exams[i]
        position[exam] = i

    # `edges` collects every conflict as a pair of chromosome indices.
    edges = []
    for pair in conflicts:
        u = pair[0]                 # first exam code, e.g. 'CSE110'
        v = pair[1]                 # second exam code, e.g. 'CSE220'
        u_index = position[u]       # its index, e.g. 0
        v_index = position[v]       # its index, e.g. 1
        edge = (u_index, v_index)
        edges.append(edge)

    return edges

In [ ]:
# ---------------------------------------------------------------------------
# 3. Random timetable generation (the initial population)
# ---------------------------------------------------------------------------

def random_chromosome(num_exams, num_slots):
    """
    Create one random candidate timetable.

    Parameters
    ----------
    num_exams : int
        Number of exams = chromosome length N. Example: 6
    num_slots : int
        Number of available slots K; each gene is an integer in 0 .. K-1. Example: 3

    Returns
    -------
    chromosome : list[int]
        A list of length num_exams; gene i is the slot assigned to exam i.
        Example: [2, 0, 1, 2, 0, 1]  (exam 0 -> slot 2, exam 1 -> slot 0, ...)
    """
    # Build the chromosome one gene at a time: each gene is a random slot.

    # TO DO
    #############################################################################
    # WRITE YOUR CODE HERE
    #############################################################################
    chromosome=[]
    for i in range(num_exams):
      chromosome.append(random.randint(0,num_slots-1))

    return chromosome


def random_population(pop_size, num_exams, num_slots):
    """
    Create a population of random candidate timetables.

    Parameters
    ----------
    pop_size : int
        How many chromosomes the population should hold. Example: 50
    num_exams : int
        Chromosome length N (number of exams). Example: 6
    num_slots : int
        Number of slots K. Example: 3

    Returns
    -------
    population : list[list[int]]
        A list of `pop_size` chromosomes.
        Example (pop_size=3): [[2, 0, 1, 2, 0, 1], [0, 0, 2, 1, 1, 0], [1, 2, 2, 0, 1, 2]]
    """
    # Build the population one chromosome at a time.
    population = []
    for _ in range(pop_size):
        chromosome = random_chromosome(num_exams, num_slots)
        population.append(chromosome)
    return population

In [ ]:
# ---------------------------------------------------------------------------
# 4. Fitness: counting clashes
# ---------------------------------------------------------------------------

def count_clashes(chromosome, edges):
    """
    Count how many conflicting pairs are scheduled in the same slot.

    This is the raw objective the algorithm drives down to 0: lower is better,
    and 0 means a perfect, conflict-free timetable.

    Parameters
    ----------
    chromosome : list[int]
        A candidate timetable; gene i is the slot of exam i.
        Example: [0, 0, 1, 1, 2, 2]
    edges : list[tuple[int, int]]
        Conflicts as index pairs (from build_edges).
        Example: [(0, 1), (0, 2), (1, 2), (2, 3), (1, 3), (3, 4), (3, 5), (4, 5)]

    Returns
    -------
    clashes : int
        The number of edges whose two exams share a slot.
        Example: chromosome [0, 0, 1, 1, 2, 2] gives 3
        (pairs (0,1), (2,3) and (4,5) collide).
    """
    # TO DO
    #############################################################################
    # WRITE YOUR CODE HERE
    #############################################################################
    c_count=0
    for i in edges:
      if chromosome[i[0]]==chromosome[i[1]]:
        c_count+=1

    return c_count

def fitness(chromosome, edges, num_edges):
    """
    Score a timetable so that higher is better (used by selection).

    Selection prefers larger values, but our objective (clashes) is something we
    want to make small. We therefore flip it: fitness = M - clashes, where M is
    the total number of conflicts. A conflict-free timetable scores the maximum, M.

    Parameters
    ----------
    chromosome : list[int]
        A candidate timetable. Example: [0, 0, 1, 1, 2, 2]
    edges : list[tuple[int, int]]
        Conflicts as index pairs. Example: [(0, 1), (0, 2), ...]
    num_edges : int
        Total number of conflicts, M = len(edges). Example: 8

    Returns
    -------
    score : int
        num_edges - clashes. Higher is better; the maximum is num_edges.
        Example: chromosome [0, 0, 1, 1, 2, 2] has 3 clashes, so score = 8 - 3 = 5.
    """
    # TO DO
    #############################################################################
    # WRITE YOUR CODE HERE
    #############################################################################
    f_score = num_edges-count_clashes(chromosome, edges)
    return f_score

In [ ]:
# ---------------------------------------------------------------------------
# 5. Selection (roulette wheel)
# ---------------------------------------------------------------------------

def roulette_wheel_selection(population, edges, num_edges):
    """
    Pick one parent using roulette-wheel (fitness-proportionate) selection.

    Imagine a wheel split into one slice per chromosome, where each slice is as wide
    as that chromosome's fitness. We pick a random point on the wheel and return the
    chromosome whose slice the point falls in. Fitter chromosomes have wider slices,
    so they are chosen more often, but every chromosome with non-zero fitness has a
    chance.

    Parameters
    ----------
    population : list[list[int]]
        The current population of chromosomes.
        Example: [[0, 1, 2, 0, 1, 2], [0, 0, 1, 1, 2, 2], ...]
    edges : list[tuple[int, int]]
        Conflicts as index pairs. Example: [(0, 1), (0, 2), ...]
    num_edges : int
        Total number of conflicts, M = len(edges). Example: 8

    Returns
    -------
    winner : list[int]
        The selected parent chromosome. Example: [0, 1, 2, 0, 1, 2]
    """
    # TO DO
    #############################################################################
    # WRITE YOUR CODE HERE
    #############################################################################
        # Step 1: work out the fitness (slice width) of every chromosome.
    # `fitness_values` example: [8, 5, 7, 6, ...]  (one number per chromosome)
    fitness_values = []
    for chromosome in population:
        value = fitness(chromosome, edges, num_edges)
        fitness_values.append(value)

    # Step 2: add up all the slice widths to get the total size of the wheel.
    # `total_fitness` example: 312
    total_fitness = 0
    for value in fitness_values:
        total_fitness += value

    # Step 3: safety check. If every chromosome has zero fitness (all conflicts
    # violated), the wheel has no width, so just pick someone at random.
    if total_fitness == 0:
        return random.choice(population)

    # Step 4: "spin" the wheel by choosing a random point between 0 and the total.
    # `spin` example: 137.4
    spin = random.uniform(0, total_fitness)

    # Step 5: walk along the wheel, adding up slice widths, until we pass `spin`.
    # The chromosome we are on at that moment is the winner.
    running_sum = 0
    for i in range(len(population)):
        running_sum += fitness_values[i]
        if running_sum >= spin:
            return population[i]

    # Step 6: safety net for tiny floating-point rounding; return the last one.
    return population[len(population) - 1]

In [ ]:
# ---------------------------------------------------------------------------
# 6. Crossover (one-point recombination)
# ---------------------------------------------------------------------------

def one_point_crossover(parent1, parent2):
    """
    Combine two parents into two children with one-point crossover.

    A cut position is chosen; child 1 takes parent 1's genes before the cut and
    parent 2's genes from the cut onward, and child 2 is the mirror image. Because
    every gene is independently a valid slot, both children are always valid
    timetables - no repair is needed.

    Parameters
    ----------
    parent1 : list[int]
        First parent timetable. Example: [0, 0, 1, 1, 2, 2]
    parent2 : list[int]
        Second parent timetable. Example: [0, 1, 2, 2, 1, 0]

    Returns
    -------
    child1, child2 : list[int]
        The two recombined offspring.
        Example with cut = 3:
            child1 = [0, 0, 1, 2, 1, 0]
            child2 = [0, 1, 2, 1, 2, 2]
    """
    # `length` is the chromosome length (same for both parents). Example: 6
    length = len(parent1)

    # `cut` is the crossover point, strictly inside the chromosome so each child
    # inherits genes from BOTH parents. Example: 3
    cut = random.randint(1, length - 1)

    # `front1` / `back2` are the two halves that make up child 1, and so on.
    front1 = parent1[:cut]    # e.g. [0, 0, 1]
    back2 = parent2[cut:]     # e.g. [2, 1, 0]
    child1 = front1 + back2   # e.g. [0, 0, 1, 2, 1, 0]

    front2 = parent2[:cut]    # e.g. [0, 1, 2]
    back1 = parent1[cut:]     # e.g. [1, 2, 2]
    child2 = front2 + back1   # e.g. [0, 1, 2, 1, 2, 2]

    return child1, child2

In [ ]:
# ---------------------------------------------------------------------------
# 7. Mutation (random resetting)
# ---------------------------------------------------------------------------

def mutate(chromosome, mutation_rate, num_slots):
    """
    Randomly reset some genes to keep the population diverse.

    Each gene is, independently and with probability `mutation_rate`, replaced by
    a fresh uniform random slot. Mutation injects new genetic material and lets the
    search escape local optima that crossover alone cannot reach.

    Parameters
    ----------
    chromosome : list[int]
        The timetable to mutate. Example: [0, 0, 1, 2, 1, 0]
    mutation_rate : float
        Per-gene probability of being reset, e.g. 0.1 (a 10% chance per gene).
    num_slots : int
        Number of slots K; a reset gene becomes a random slot in 0 .. K-1. Example: 3

    Returns
    -------
    mutated : list[int]
        A new chromosome with zero or more genes changed (the input is untouched).
        Example: [2, 0, 1, 2, 1, 0]  (gene 0 was reset from 0 to 2)
    """
    # Work on a copy so the caller's chromosome is left unchanged.
    mutated = chromosome[:]

    for i in range(len(mutated)):
        chance = random.random()          # a random number in [0, 1), e.g. 0.07
        if chance < mutation_rate:
            new_slot = random.randint(0, num_slots - 1)   # a fresh random slot
            mutated[i] = new_slot

    return mutated

In [ ]:
# ---------------------------------------------------------------------------
# 8. Small helpers for the main loop (find the best, and the elites)
# ---------------------------------------------------------------------------

def find_best(population, edges):
    """
    Find the single best timetable in a population (the one with fewest clashes).

    Parameters
    ----------
    population : list[list[int]]
        The population to search. Example: [[0, 1, 2, 0, 1, 2], [0, 0, 1, 1, 2, 2], ...]
    edges : list[tuple[int, int]]
        Conflicts as index pairs. Example: [(0, 1), (0, 2), ...]

    Returns
    -------
    best : list[int]
        The chromosome with the fewest clashes. Example: [1, 2, 0, 1, 0, 2]
    best_clashes : int
        That chromosome's clash count. Example: 0
    """
    # Start by assuming the first chromosome is the best, then look for better ones.
    best = population[0]
    best_clashes = count_clashes(best, edges)

    for chromosome in population:
        clashes = count_clashes(chromosome, edges)
        if clashes < best_clashes:
            best = chromosome
            best_clashes = clashes

    return best, best_clashes


def get_elites(population, edges, elite_count):
    """
    Return copies of the `elite_count` best timetables in the population.

    These elites are carried into the next generation unchanged (elitism), so the
    best solution found so far is never lost to crossover or mutation.

    Parameters
    ----------
    population : list[list[int]]
        The current population. Example: [[0, 1, 2, 0, 1, 2], [0, 0, 1, 1, 2, 2], ...]
    edges : list[tuple[int, int]]
        Conflicts as index pairs. Example: [(0, 1), (0, 2), ...]
    elite_count : int
        How many top chromosomes to return. Example: 2

    Returns
    -------
    elites : list[list[int]]
        Copies of the `elite_count` chromosomes with the fewest clashes.
        Example: [[1, 2, 0, 1, 0, 2], [0, 1, 2, 0, 1, 1]]
    """
    # `scored` pairs each chromosome's clash count with its position in the list.
    # Sorting these pairs orders them from fewest clashes to most.
    # Example: [(0, 7), (1, 3), (1, 12), (2, 0), ...]
    scored = []
    for i in range(len(population)):
        clashes = count_clashes(population[i], edges)
        scored.append((clashes, i))

    # Sort by clash count (the first item of each pair); ties fall back to the
    # index, which keeps the order predictable.
    scored.sort()

    # Take the indices of the best `elite_count` chromosomes and copy them.
    elites = []
    how_many = min(elite_count, len(scored))
    for k in range(how_many):
        index_of_elite = scored[k][1]
        elite_copy = population[index_of_elite][:]
        elites.append(elite_copy)

    return elites

In [ ]:
# ---------------------------------------------------------------------------
# 9. The genetic algorithm (the main evolutionary loop)
# ---------------------------------------------------------------------------

def genetic_algorithm(edges, num_exams, num_slots,
                      pop_size=50, generations=200,
                      crossover_rate=0.9, mutation_rate=0.1,
                      elite_count=2):
    """
    Evolve a population of timetables until one is conflict-free (or time runs out).

    Parameters
    ----------
    edges : list[tuple[int, int]]
        Conflicts as index pairs. Example: [(0, 1), (0, 2), ...]
    num_exams : int
        Chromosome length N. Example: 6
    num_slots : int
        Number of slots K. Example: 3
    pop_size : int
        Number of chromosomes kept each generation. Example: 50
    generations : int
        Maximum number of generations to run. Example: 200
    crossover_rate : float
        Probability a selected pair is recombined (otherwise copied). Example: 0.9
    mutation_rate : float
        Per-gene mutation probability. Example: 0.1
    elite_count : int
        How many best chromosomes are carried over unchanged each generation
        (elitism). Example: 2

    Returns
    -------
    best : list[int]
        The best timetable found. Example: [1, 2, 0, 1, 0, 2]
    best_clashes : int
        Its clash count (0 if a conflict-free timetable was found). Example: 0
    """
    # `num_edges` is the total number of conflicts, M. Example: 8
    num_edges = len(edges)

    # `population` starts as random timetables.
    population = random_population(pop_size, num_exams, num_slots)

    # `best` / `best_clashes` track the best timetable seen so far across ALL
    # generations. They are initialised from the starting population.
    best, best_clashes = find_best(population, edges)

    for generation in range(generations):
        # Stop early the moment a perfect, conflict-free timetable exists.
        if best_clashes == 0:
            break

        # Elitism: the next generation starts with copies of the best chromosomes.
        # `next_population` example (elite_count=2): [[1, 2, 0, 1, 0, 2], [0, 1, 2, 0, 1, 1]]
        next_population = get_elites(population, edges, elite_count)

        # Fill the rest of the next generation with offspring.
        while len(next_population) < pop_size:
            # Choose two parents with roulette-wheel selection.
            parent1 = roulette_wheel_selection(population, edges, num_edges)
            parent2 = roulette_wheel_selection(population, edges, num_edges)

            # Recombine with probability crossover_rate, otherwise copy the parents.
            chance = random.random()
            if chance < crossover_rate:
                child1, child2 = one_point_crossover(parent1, parent2)
            else:
                child1 = parent1[:]
                child2 = parent2[:]

            # Mutate both children.
            child1 = mutate(child1, mutation_rate, num_slots)
            child2 = mutate(child2, mutation_rate, num_slots)

            # Add the children to the next generation (without overshooting pop_size).
            next_population.append(child1)
            if len(next_population) < pop_size:
                next_population.append(child2)

        # The next generation becomes the current population.
        population = next_population

        # Update the running best with this generation's strongest chromosome.
        generation_best, generation_best_clashes = find_best(population, edges)
        if generation_best_clashes < best_clashes:
            best = generation_best[:]
            best_clashes = generation_best_clashes

    return best, best_clashes

In [ ]:
# ---------------------------------------------------------------------------
# 10. Formatting the result
# ---------------------------------------------------------------------------

def format_solution(exams, chromosome, clashes):
    """
    Turn a chromosome into the required output text.

    The output is one line per exam ("EXAM_CODE slot"), followed by a final line
    "Clashes: c".

    Parameters
    ----------
    exams : list[str]
        Exam codes in order; exams[i] is the exam at index i.
        Example: ['CSE110', 'CSE220', 'MAT120', 'PHY101', 'ENG101', 'ECO101']
    chromosome : list[int]
        The chosen timetable; gene i is the slot of exam i.
        Example: [1, 2, 0, 1, 0, 2]
    clashes : int
        The clash count of that timetable. Example: 0

    Returns
    -------
    text : str
        The formatted schedule. Example:
            "CSE110 1\\nCSE220 2\\nMAT120 0\\nPHY101 1\\nENG101 0\\nECO101 2\\nClashes: 0"
    """
    # `rows` is one "exam slot" line per exam.
    rows = []
    for i in range(len(exams)):
        exam = exams[i]                       # e.g. 'CSE110'
        slot = chromosome[i]                  # e.g. 1
        line = exam + " " + str(slot)         # e.g. 'CSE110 1'
        rows.append(line)

    # The final line reports the clash count.
    rows.append("Clashes: " + str(clashes))

    # Join all the lines into one block of text separated by newlines.
    text = "\n".join(rows)
    return text

In [ ]:
# ---------------------------------------------------------------------------
# 11. Driver
# ---------------------------------------------------------------------------

# The sample input, exactly as it appears in the problem statement.
SAMPLE_INPUT = """
6 8
3
CSE110
CSE220
MAT120
PHY101
ENG101
ECO101
CSE110 CSE220
CSE110 MAT120
CSE220 MAT120
MAT120 PHY101
CSE220 PHY101
PHY101 ENG101
PHY101 ECO101
ENG101 ECO101
"""


def main():
    """Parse the input, run the GA, and print the schedule and its clash count."""
    # A fixed seed makes this faculty solution reproducible. The GA is stochastic,
    # so a different seed may yield a different (but equally valid) 0-clash schedule.
    random.seed(42)

    # `exams`, `conflicts`, `num_slots` come straight from the input.
    exams, conflicts, num_slots = parse_input(SAMPLE_INPUT)

    # `edges` is the conflict list translated into chromosome-index pairs.
    edges = build_edges(exams, conflicts)

    # Run the search. `best` is the winning timetable, `best_clashes` its clash count.
    best, best_clashes = genetic_algorithm(edges, len(exams), num_slots)

    # Print the schedule in the required output format.
    output = format_solution(exams, best, best_clashes)
    print(output)


if __name__ == "__main__":
    main()

TypeError: 'NoneType' object is not subscriptable